**GOAL** : This section highlights the characteristics / aspects of a variable that should be taken into account before engineering more features / building the final predictive model



Characteristics
* missing data -> effect of data with missing values on the target (this will determine the kind of imputation techniques)
* variables containing string values (cardinality, string frequency) -> how to better encode values to numbers so that we can input them to mathematical models
* linear model assumption -> can the data be modelled using linear algorithms (linear / logistic regression) or (decision trees, ensemble models)
* distributions
* outliers -> what data is highly unexpected
* magnitude -> scale of data feature with respect to other features




## Missing data


Any value of a feature that is missing.

* common occurence
* a value can be lost due to human / machine error
* the value simply does not exisits -> needs to be derived may be (feature engineering comes here)
* some which we cannot derive we give a common value of 'NA'


Impacts of missing data
* Imputation may distort the variable distribution hence affecting ml / deep learning model performance.

Types of events that lead to missing data
* Missing data completly at random (MCAR) -> uniformly possible for a value to be missing disregarding them will not cause any bias in the model
* Missing data at random (MAR) -> the possibility of a data value being missing is completly dependant on available information
* Missing data not at random (MNAR) -> the probability of the data value bieng missing is dependant on a more defined mechanisim

Knowledge of data collection leads to understanding why some data is missing in the data set also provides the knowledge and intuituion required to engineer new features.

In [6]:
from pathlib import Path

import pandas as pd
import numpy as np

In [7]:
data_dir = Path().resolve().parent / "data"

loan_data_path  = data_dir / "loan.csv"
titanic_data_path = data_dir / "titanic.csv"


df_loan = pd.read_csv(loan_data_path)
df_titanic = pd.read_csv(titanic_data_path)

In [8]:
df_titanic.columns

Index(['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket',
       'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest'],
      dtype='object')

In [9]:
df_titanic.isnull().sum()

pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

In [10]:
df_titanic.isnull().mean()

pclass       0.000000
survived     0.000000
name         0.000000
sex          0.000000
age          0.200917
sibsp        0.000000
parch        0.000000
ticket       0.000000
fare         0.000764
cabin        0.774637
embarked     0.001528
boat         0.628724
body         0.907563
home.dest    0.430863
dtype: float64

In [11]:
df_titanic.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [17]:
# hypothesis cabin data is missing because the data was captured after the crash of titanic happened hence not surviving people couldn't provide their cabin information
# if cabin is missing we would expect that the person is dead
# hence for non-surviving data there would be high possibility that the cabin information is missing
# to form this hypothesis we need to know how the data was collected.


df_titanic['cabin_null'] = np.where(df_titanic['cabin'].isnull(), 1, 0)
df_titanic.groupby('survived')['cabin_null'].mean()

survived
0    0.873918
1    0.614000
Name: cabin_null, dtype: float64

In [18]:
# lets try the same for age
# for non-surviving people there is no way we would know thier age.

df_titanic['age_null'] = np.where(df_titanic['age'].isnull(), 1, 0)
df_titanic.groupby('survived')['age_null'].mean()

survived
0    0.234858
1    0.146000
Name: age_null, dtype: float64

In [19]:
# lets look at the column of embarked it has the lowest number of missing values on average
df_titanic[df_titanic['embarked'].isnull()]

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest,cabin_null,age_null
168,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN,6,NaN,NaN,0,0
284,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN,6,NaN,"Cincinatti, OH",0,0


In [20]:
# one common thing we can se is that the fare body sex is the same 
# lets given that only two such rows are present with missing values
# we can be rest assured that this is a genuine error meaning this data is missing due to MCAR (completly at random) might be lost due to some machine error or human error
# we can show this that other rows with sex as female have a embarked data point

In [28]:
# lets look at the loan data now

df_loan.isnull().mean()

customer_id               0.0000
disbursed_amount          0.0000
interest                  0.0000
market                    0.0000
employment                0.0611
time_employed             0.0529
householder               0.0000
income                    0.0000
date_issued               0.0000
target                    0.0000
loan_purpose              0.0000
number_open_accounts      0.0000
date_last_payment         0.0000
number_credit_lines_12    0.9762
dtype: float64

In [29]:
# lets look at employment and time_employed information
# one relation between these two is that these both are provided by the borrower at the time applying for loan

# lets look at the unique values for time employed and employement
# it is assumed both have na values of around 6%

df_loan['employment'].unique(), df_loan['time_employed'].unique()

(array(['Teacher', 'Accountant', 'Statistician', 'Other', 'Bus driver',
        'Secretary', 'Software developer', 'Nurse', 'Taxi driver', nan,
        'Civil Servant', 'Dentist'], dtype=object),
 array(['<=5 years', '>5 years', nan], dtype=object))

In [32]:
# we can assume nan means the person was not having any sort of employment or was self-employed or was retired hence at the point of applying for the loan the value was left empty

# this also means that it is possible if they are not employed they wouldn't have had the time employed information at the time of applying for the loan

# this can be infered by doing the inversion
# lets look at the proportion of missing values for time_employed when an employement value was provided

df_loan[~df_loan['employment'].isnull()]['time_employed'].isnull().mean()

np.float64(0.0005325380764724678)

In [33]:
# we can see that the proportion of missing value is 0 till the 4th digit of precision hence it highly unlikely that while providing the information time_employed is left null if a person has a active job

In [34]:
# we can look at the direct correlation too
df_loan[df_loan['employment'].isnull()]['time_employed'].isnull().mean()

np.float64(0.8576104746317512)

In [ ]:
# it is can be seen that it is 85 % likely that if employment is not provided the time employed column is left empty
# hence this is a case of MAR -> missing at random (dependant on a other feature column)